# Handling missing values: detect, choose, impute (without leaking)

_Data Wrangling_

**Find the holes, figure out WHY they are there, then drop or fill — fit on train only.**

A blank cell is not nothing &mdash; it is a small mystery. The whole craft of handling missing
       values is: (1) measure how many blanks there are and where, (2) reason about
       why they are blank, and (3) pick drop-or-fill accordingly &mdash; while never
       letting the test set influence the fill.

       Detect. The one-line summary is df.isna().sum() &mdash; the number of
       missing cells per column. Divide by len(df) for a percentage. The
       missingno library draws this: a matrix plot where blanks show as white streaks, so
       you can see whether two columns go missing together.

---

This notebook builds the missing-value workflow one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — pandas + scikit-learn

We walk the full missing-value workflow in four steps: (1) inject reproducible blanks into a real frame, (2) **detect** how much is missing and where, (3) split first then weigh **drop vs. impute**, and (4) bundle imputer + model in a leak-proof Pipeline.

### Step 1 — Load a real frame and inject reproducible blanks

We start from the Wine dataset and knock holes in two columns with a seeded random generator, so the missingness is the same every run. `magnesium` loses 30% of its cells and `proline` 12% — enough to make the detect-and-fill choices matter.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.experimental import enable_iterative_imputer   # noqa: needed to expose IterativeImputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

# Load a real frame; copy X so we can punch holes without touching the original.
data = load_wine(as_frame=True)
X = data.data.copy()
y = data.target

# Inject reproducible missingness into two columns.
rng = np.random.RandomState(42)
for col, frac in [('magnesium', 0.30), ('proline', 0.12)]:
    n_missing = int(frac * len(X))
    idx = rng.choice(len(X), n_missing, replace=False)
    X.loc[idx, col] = np.nan

### Step 2 — Detect: count the blanks per column

The one-line summary is `df.isna().sum()` — the number of missing cells per column. Dividing by `len(df)` turns that into a percentage, which is the number you actually reason about. The `missingno` library draws the same thing visually, so you can spot whether two columns go blank together.

In [ ]:
# Missing CELLS per column.
missing_counts = X.isna().sum()
print(missing_counts)

# Missing PERCENT per column.
missing_percent = (X.isna().mean() * 100).round(1)
print(missing_percent)

# Visual: import missingno as msno; msno.matrix(X); msno.bar(X)
#   -> white streaks show WHERE blanks fall and whether columns go missing together.

### Step 3 — Split first, then weigh drop vs. impute

Split **before** touching the blanks so nothing is learned from the test rows. Then two options: **drop** (only safe when the missingness is MCAR or the fraction is tiny), or **impute** — and crucially fit the imputer on the training rows only, with `add_indicator=True` so the model still sees a "was missing" flag.

In [ ]:
# Split FIRST so nothing is learned from the test rows.
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

# Option A: DROP (only safe when missingness is MCAR / fraction is tiny).
X_tr.dropna()                               # drop rows with any blank
X_tr.dropna(axis=1)                         # drop columns with any blank
X.dropna(thresh=int(0.5 * len(X)), axis=1)  # drop columns >50% missing

# Option B: IMPUTE, fit on TRAIN ONLY, with a missingness indicator.
median_imp = SimpleImputer(strategy='median', add_indicator=True).fit(X_tr)
knn_imp    = KNNImputer(n_neighbors=5, add_indicator=True).fit(X_tr)
iter_imp   = IterativeImputer(random_state=0, add_indicator=True).fit(X_tr)
# .transform(X_te) now uses ONLY the train-learned medians / neighbours / model.

### Step 4 — Bundle impute + model in a leak-proof Pipeline

The clean way to avoid leakage in cross-validation is to put the imputer and the classifier in a single `Pipeline`. Then `cross_val_score` re-fits the imputer inside each fold on that fold's training rows only — so no test information ever bleeds into the fill. The time-series fills (`ffill`/`bfill`) are a separate idiom for time-indexed frames.

In [ ]:
# Bundle impute + model so CV re-fits the imputer per fold (no leakage).
pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True)),  # 13 feats + 2 indicators
    ('clf', RandomForestClassifier(random_state=0)),
])

cv_accuracy = cross_val_score(pipe, X, y, cv=5).mean().round(3)
print('CV accuracy:', cv_accuracy)

pipe.fit(X_tr, y_tr)
test_accuracy = round(pipe.score(X_te, y_te), 3)
print('test accuracy:', test_accuracy)

# Time-series fills (separate idiom, for a time-indexed frame):
# ts = ts.ffill()   # carry last observed value forward across a gap
# ts = ts.bfill()   # pull the next value backward (e.g. at the start)

## Visualize the data & results

_Inject missingness into two columns of the Wine dataset, then (1) show missing-% per column, and (2) show what median-imputing 'magnesium' does to its distribution — a spike appears at the median._

### Step 1 — Rebuild the same missingness on four columns

We take four Wine columns and inject the identical seeded blanks (30% of `magnesium`, 12% of `proline`) so this view matches the run above. Printing the missing percent per column confirms the holes landed where we expect.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.impute import SimpleImputer

# Four columns from the real Wine frame.
df = load_wine(as_frame=True).frame[
    ['alcohol', 'magnesium', 'proline', 'color_intensity']].copy()
n = len(df)                                   # 178

# Inject the same seeded missingness as before.
rng = np.random.RandomState(42)
miss = df.copy()
miss.loc[rng.choice(n, int(0.30 * n), replace=False), 'magnesium'] = np.nan
miss.loc[rng.choice(n, int(0.12 * n), replace=False), 'proline']   = np.nan

# Missing % per column.
missing_percent = (miss.isna().mean() * 100).round(1)
print(missing_percent)                        # magnesium 29.8, proline 11.8

### Step 2 — Show what median imputation does to the distribution

We bin `magnesium` into fixed bins and compare the histogram of the **observed** values against the same column **after** median imputation. Every filled cell lands on the single median value, so one bin gains a tall spike — the visual signature of mean/median imputation shrinking the variance.

In [ ]:
# Fixed bins so the before/after histograms are directly comparable.
bins = np.linspace(70, 165, 8)                # 7 fixed bins

# The observed-only median is what SimpleImputer will use as the fill value.
med = miss['magnesium'].median()              # 98.0  (observed only)

# After median imputation, every blank becomes this median.
after = SimpleImputer(strategy='median').fit_transform(miss[['magnesium']]).ravel()

# Histogram counts before vs. after.
before_counts = np.histogram(miss['magnesium'].dropna(), bins=bins)[0]
after_counts  = np.histogram(after, bins=bins)[0]

print('edges :', np.round(bins).astype(int))  # [ 70 84 97 111 124 138 151 165]
print('before:', before_counts)               # [ 6 53 37 20  6  2  1]
print('after :', after_counts)                # [ 6 53 90 20  6  2  1]  -> spike at 97-111

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your training frame has a numeric column that is 35% missing and clearly right-skewed (a few huge values). A teammate proposes SimpleImputer(strategy='mean') applied to the whole dataset before splitting. Name the two separate problems and fix each.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Spot the distribution problem: filling 35% of a skewed column with the mean stacks a spike at a value pulled up by the outliers. — _Mean-imputation flattens the distribution and shrinks variance by roughly the missing fraction; the mean of a skewed column is not representative._
- Switch to strategy='median' and add add_indicator=True. — _The median is robust to the big values, and the indicator keeps the "was missing" signal in case the column is MNAR._
- Spot the leakage problem: fitting before the split learns the fill value from test rows too. — _That bleeds test information into training and inflates measured performance._
- Move the imputer into a Pipeline and split first, so it is fit on train only. — _The imputer's learned median (and neighbours, for KNN) then comes only from the training split._

**Answer:** Two issues. (1) Mean on a skewed column stacks a spike and shrinks variance &mdash; use strategy='median' and add a missingness indicator. (2) Fitting before the split leaks &mdash; put the imputer in a Pipeline and split first so it is fit on the training set only.

</details>

**Problem 2.** Two columns: "income" on a voluntary form (high earners tend to skip it) and "temperature" from a sensor that drops about 1% of readings uniformly at random. Classify each mechanism and say whether dropping the affected rows is safe.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For income, ask whether the blank depends on the missing value itself. — _High earners skip BECAUSE income is high &mdash; the blank depends on the unobserved value, so it is MNAR (Missing Not At Random)._
- For temperature, ask the same. — _A uniform 1% sensor dropout depends on nothing &mdash; it is MCAR (Missing Completely At Random)._
- Decide drop vs keep per mechanism. — _Dropping MCAR rows is unbiased; dropping MNAR rows would systematically remove high earners and bias the sample._

**Answer:** Income is MNAR &mdash; the blank depends on the value itself. Dropping those rows would drop the high earners and bias the sample; impute and keep a missingness indicator. Temperature is MCAR &mdash; the dropout is unrelated to anything; dropping the 1% of rows is safe (the survivors are a fair sample).

</details>

**Problem 3.** You have a daily revenue time series with a few isolated missing days, and a separate wide customer table where one column is MAR (its blanks are predictable from age and region). Which imputation fits each?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- For the time series, use the value adjacent in time. — _df.ffill() carries the last known day forward (or bfill near the start) &mdash; the nearest day is the best guess for a one-day gap._
- For the MAR column, use the other columns. — _KNNImputer (or IterativeImputer) fills from similar rows; since the blanks are explained by age and region, a multivariate fill recovers them well._
- Fit both on train only. — _Forward-fill across a train/test boundary and the neighbour search both leak if computed on all the data._

**Answer:** Time series: ffill / bfill &mdash; the temporally adjacent value is the natural fill. MAR customer column: KNNImputer or IterativeImputer, which exploit the other columns that explain the missingness. Both must be fit on the training data only.

</details>